# Week 6: Data Cleaning & Validation + GitHub & Version Control

This week has two parts. First, you'll learn to clean messy data — duplicates, missing values,
bad types, inconsistent strings, and outliers. These are skills you'll use on every real dataset.
Second, you'll connect Databricks to GitHub so your notebooks are backed up and shareable.

## Part 1: Finding and Handling Duplicates

Duplicate rows can inflate counts and skew averages. The first step in any cleaning pipeline is
checking for them.

### SQL: Find Duplicates with GROUP BY ... HAVING

In [0]:
%sql
-- Find duplicate flight numbers on the same day
SELECT year, month, day, carrier, flight, COUNT(*) AS num_records
FROM flights
GROUP BY year, month, day, carrier, flight
HAVING COUNT(*) > 1
ORDER BY num_records DESC
LIMIT 10

year,month,day,carrier,flight,num_records
2013,8,3,WN,2269,2
2013,7,13,WN,2269,2
2013,7,20,WN,2269,2
2013,7,27,WN,2269,2
2013,6,15,WN,2269,2
2013,7,6,WN,2269,2
2013,6,8,WN,2269,2
2013,6,29,WN,2269,2
2013,6,22,WN,2269,2
2013,8,10,WN,2269,2


### SQL: Remove Duplicates with ROW_NUMBER()

This pattern from Week 4 assigns a row number within each group — keep row 1, discard the rest:

In [0]:
%sql
-- Keep one record per flight per day
WITH numbered AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY year, month, day, carrier, flight
               ORDER BY sched_dep_time
           ) AS rn
    FROM flights
)
SELECT *
FROM numbered
WHERE rn = 1
LIMIT 10

rownames,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,rn
658,2013,1,1,1825,1829,-4,2056,2053,3,9E,3286,N906XJ,JFK,DTW,107,509,18,29,2013-01-01T23:00:00.000Z,1
429,2013,1,1,1452,1455,-3,1637,1639,-2,9E,3295,N920XJ,JFK,BUF,68,301,14,55,2013-01-01T19:00:00.000Z,1
754,2013,1,1,2015,2005,10,2149,2144,5,9E,3320,N931XJ,JFK,BUF,62,301,20,5,2013-01-02T01:00:00.000Z,1
557,2013,1,1,1637,1545,52,1858,1819,39,9E,3321,N604LR,JFK,MSP,173,1029,15,45,2013-01-01T20:00:00.000Z,1
726,2013,1,1,1939,1840,59,29,2151,null,9E,3325,N905XJ,JFK,DFW,null,1391,18,40,2013-01-01T23:00:00.000Z,1
506,2013,1,1,1554,1600,-6,1701,1734,-33,9E,3331,N931XJ,JFK,BOS,41,187,16,0,2013-01-01T21:00:00.000Z,1
496,2013,1,1,1546,1540,6,1753,1748,5,9E,3338,N904XJ,JFK,ORD,146,740,15,40,2013-01-01T20:00:00.000Z,1
802,2013,1,1,2115,1700,255,2330,1920,250,9E,3347,N924XJ,JFK,CVG,115,589,17,0,2013-01-01T22:00:00.000Z,1
762,2013,1,1,2023,1945,38,2240,2206,34,9E,3352,N602LR,JFK,CVG,118,589,19,45,2013-01-02T00:00:00.000Z,1
778,2013,1,1,2046,2035,11,2144,2213,-29,9E,3357,N916XJ,JFK,DCA,43,213,20,35,2013-01-02T01:00:00.000Z,1


### Python: Drop Duplicates

`.dropDuplicates()` with no arguments removes rows where every column matches. Passing a list
of columns keeps the first occurrence for each unique combination of those columns.

In [0]:
flights = spark.table("flights")

# Count before dedup
print(f"Before: {flights.count()} rows")

# Drop exact duplicate rows (all columns must match)
deduped = flights.dropDuplicates()
print(f"After dropDuplicates(): {deduped.count()} rows")

# Drop duplicates based on specific columns
deduped_by_key = flights.dropDuplicates(["year", "month", "day", "carrier", "flight"])
print(f"After dropDuplicates([keys]): {deduped_by_key.count()} rows")

Before: 336776 rows
After dropDuplicates(): 336776 rows
After dropDuplicates([keys]): 336752 rows


## Part 2: Handling Missing Values

Week 3 introduced NULL handling basics. Now we'll go deeper — auditing missing data across all
columns and deciding how to fill gaps.

### SQL: NULL Audit

In [0]:
%sql
-- Count and percentage of missing values for key columns
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN dep_delay IS NULL THEN 1 ELSE 0 END) AS missing_dep_delay,
    ROUND(100.0 * SUM(CASE WHEN dep_delay IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_dep_delay,
    SUM(CASE WHEN arr_delay IS NULL THEN 1 ELSE 0 END) AS missing_arr_delay,
    ROUND(100.0 * SUM(CASE WHEN arr_delay IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_arr_delay,
    SUM(CASE WHEN air_time IS NULL THEN 1 ELSE 0 END) AS missing_air_time,
    ROUND(100.0 * SUM(CASE WHEN air_time IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_air_time
FROM flights

total_rows,missing_dep_delay,pct_dep_delay,missing_arr_delay,pct_arr_delay,missing_air_time,pct_air_time
336776,8255,2.45,9430,2.80,9430,2.80


### SQL: Fill with Median Using a CTE

In [0]:
%sql
-- Fill missing arr_delay with the median value
WITH median_val AS (
    SELECT PERCENTILE(arr_delay, 0.5) AS median_delay
    FROM flights
    WHERE arr_delay IS NOT NULL
)
SELECT
    carrier, flight, arr_delay,
    COALESCE(arr_delay, median_delay) AS arr_delay_filled
FROM flights, median_val
LIMIT 15

carrier,flight,arr_delay,arr_delay_filled
UA,1545,11,11.0
UA,1714,20,20.0
AA,1141,33,33.0
B6,725,-18,-18.0
DL,461,-25,-25.0
UA,1696,12,12.0
B6,507,19,19.0
EV,5708,-14,-14.0
B6,79,-8,-8.0
AA,301,8,8.0


### Python: Drop and Fill Missing Values

In [0]:
from pyspark.sql.functions import col, count, when, round as spark_round, mean

# NULL audit — count missing per column
flights = spark.table("flights")

flights.select(
    count("*").alias("total_rows"),
    count(when(col("dep_delay").isNull(), 1)).alias("missing_dep_delay"),
    count(when(col("arr_delay").isNull(), 1)).alias("missing_arr_delay"),
    count(when(col("air_time").isNull(), 1)).alias("missing_air_time")
).show()

+----------+-----------------+-----------------+----------------+
|total_rows|missing_dep_delay|missing_arr_delay|missing_air_time|
+----------+-----------------+-----------------+----------------+
|    336776|             8255|             9430|            9430|
+----------+-----------------+-----------------+----------------+



In [0]:
# Drop rows missing arr_delay
flights_clean = flights.dropna(subset=["arr_delay"])
print(f"Before: {flights.count()}, After dropna: {flights_clean.count()}")

Before: 336776, After dropna: 327346


In [0]:
# Fill missing arr_delay with the mean
mean_delay = flights.select(mean("arr_delay")).collect()[0][0]
print(f"Mean arr_delay: {mean_delay:.2f}")

flights_filled = flights.fillna({"arr_delay": mean_delay})
display(flights_filled.select("carrier", "flight", "arr_delay").limit(10))

Mean arr_delay: 6.90


carrier,flight,arr_delay
UA,1545,11
UA,1714,20
AA,1141,33
B6,725,-18
DL,461,-25
UA,1696,12
B6,507,19
EV,5708,-14
B6,79,-8
AA,301,8


### When to Drop vs Fill vs Leave NULL

| Situation | Strategy | Example |
|-----------|----------|---------|
| Small % missing, random pattern | Fill with mean or median | A few flights missing delay values |
| Large % missing (>30%) | Leave NULL or drop column | A column mostly empty |
| Missing = meaningful (e.g., cancelled) | Leave NULL or create indicator | Cancelled flights have NULL delay |
| Critical analysis column | Drop rows | Can't compute delay stats without delay |
| Categorical variable | Fill with mode or "Unknown" | Missing carrier name |

## Part 3: Data Type Validation and Casting

Wrong data types cause silent errors — a number stored as a string won't sort or aggregate correctly.

### SQL: Inspect and Cast Types

In [0]:
%sql
-- Check the schema of flights
DESCRIBE flights

col_name,data_type,comment
rownames,int,null
year,int,null
month,int,null
day,int,null
dep_time,int,null
sched_dep_time,int,null
dep_delay,int,null
arr_time,int,null
sched_arr_time,int,null
arr_delay,int,null


In [0]:
%sql
-- Cast string to integer (if dep_time were stored as string)
SELECT
    carrier,
    dep_time,
    CAST(dep_time AS INT) AS dep_time_int
FROM flights
LIMIT 10

carrier,dep_time,dep_time_int
UA,517,517
UA,533,533
AA,542,542
B6,544,544
DL,554,554
UA,554,554
B6,555,555
EV,557,557
B6,557,557
AA,558,558


In [0]:
%sql
-- Validate categorical values — what distinct values exist?
SELECT DISTINCT cut FROM diamonds ORDER BY cut

cut
Fair
Good
Ideal
Premium
Very Good


In [0]:
%sql
-- Check for unexpected values in a categorical column
SELECT DISTINCT carrier FROM flights ORDER BY carrier

carrier
9E
AA
AS
B6
DL
EV
F9
FL
HA
MQ


### Python: Inspect and Cast Types

In [0]:
# Check the schema
flights = spark.table("flights")
flights.printSchema()

root
 |-- rownames: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- dep_time: integer (nullable = true)
 |-- sched_dep_time: integer (nullable = true)
 |-- dep_delay: integer (nullable = true)
 |-- arr_time: integer (nullable = true)
 |-- sched_arr_time: integer (nullable = true)
 |-- arr_delay: integer (nullable = true)
 |-- carrier: string (nullable = true)
 |-- flight: integer (nullable = true)
 |-- tailnum: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- air_time: integer (nullable = true)
 |-- distance: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- minute: integer (nullable = true)
 |-- time_hour: timestamp (nullable = true)



In [0]:
# Cast a column to a different type
from pyspark.sql.types import IntegerType

flights_cast = flights.withColumn("dep_time_int", col("dep_time").cast(IntegerType()))
display(flights_cast.select("carrier", "dep_time", "dep_time_int").limit(10))

carrier,dep_time,dep_time_int
UA,517,517
UA,533,533
AA,542,542
B6,544,544
DL,554,554
UA,554,554
B6,555,555
EV,557,557
B6,557,557
AA,558,558


In [0]:
# Check distinct values in a categorical column
diamonds = spark.table("diamonds")
display(diamonds.select("cut").distinct().orderBy("cut"))

cut
Fair
Good
Ideal
Premium
Very Good


In [0]:
# Check for invalid values — are there any cuts not in the expected list?
valid_cuts = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
invalid = diamonds.filter(~col("cut").isin(valid_cuts))
print(f"Invalid cut values: {invalid.count()}")

Invalid cut values: 0


## Part 4: String Cleaning

Real-world text data is messy — extra spaces, inconsistent capitalization, special characters.
Here are the tools to fix it.

### Create a Messy Dataset for Practice

In [0]:
%sql
-- Create a temp view with messy name data
CREATE OR REPLACE TEMP VIEW messy_names AS
SELECT * FROM VALUES
    ('  John Smith  ', 'new york', '  john.smith@EMAIL.COM  '),
    ('JANE   DOE', 'New York', 'jane.doe@email.com'),
    ('bob johnson', 'NEW YORK', '  BOB.JOHNSON@email.com'),
    ('  Alice Williams', 'los angeles', 'alice.w@Email.Com  '),
    ('CHARLIE brown  ', 'Los Angeles', 'charlie@EMAIL.COM')
AS t(name, city, email)


Executing subquery: -- Fill missing arr_delay with the median value
WITH median_val AS (
    SELECT PERCENTILE(arr_delay, 0.5) AS median_delay
    FROM flights
    WHERE arr_delay IS NOT NULL
)
SELECT
    carrier, flight, arr_delay,
    COALESCE(arr_delay, median_delay) AS arr_delay_filled
FROM flights, median_val
LIMIT 15.
Executing subquery: -- Check the schema of flights
DESCRIBE flights.
Executing subquery: -- Cast string to integer (if dep_time were stored as string)
SELECT
    carrier,
    dep_time,
    CAST(dep_time AS INT) AS dep_time_int
FROM flights
LIMIT 10.
Executing subquery: -- Validate categorical values — what distinct values exist?
SELECT DISTINCT cut FROM diamonds ORDER BY cut.
Executing subquery: -- Check for unexpected values in a categorical column
SELECT DISTINCT carrier FROM flights ORDER BY carrier.
Executing subquery: -- Create a temp view with messy name data
CREATE OR REPLACE TEMP VIEW messy_names AS
SELECT * FROM VALUES
    ('  John Smith  ', 'new york', ' 

### SQL: String Functions

In [0]:
%sql
-- Clean the messy names
SELECT
    name AS original,
    TRIM(name) AS trimmed,
    UPPER(TRIM(name)) AS upper_name,
    LOWER(TRIM(name)) AS lower_name,
    INITCAP(TRIM(name)) AS proper_name
FROM messy_names

original,trimmed,upper_name,lower_name,proper_name
John Smith,John Smith,JOHN SMITH,john smith,John Smith
JANE DOE,JANE DOE,JANE DOE,jane doe,Jane Doe
bob johnson,bob johnson,BOB JOHNSON,bob johnson,Bob Johnson
Alice Williams,Alice Williams,ALICE WILLIAMS,alice williams,Alice Williams
CHARLIE brown,CHARLIE brown,CHARLIE BROWN,charlie brown,Charlie Brown


In [0]:
%sql
-- Remove extra internal spaces and clean email
SELECT
    INITCAP(TRIM(REGEXP_REPLACE(name, '\\s+', ' '))) AS clean_name,
    LOWER(TRIM(email)) AS clean_email,
    INITCAP(TRIM(city)) AS clean_city
FROM messy_names

clean_name,clean_email,clean_city
John Smith,john.smith@email.com,New York
Jane Doe,jane.doe@email.com,New York
Bob Johnson,bob.johnson@email.com,New York
Alice Williams,alice.w@email.com,Los Angeles
Charlie Brown,charlie@email.com,Los Angeles


In [0]:
%sql
-- Clean airline names from the airlines table
SELECT
    carrier,
    name,
    TRIM(REGEXP_REPLACE(name, ' (Inc\\.|Co\\.|Corporation|Airlines|Air Lines)', '')) AS short_name
FROM airlines

carrier,name,short_name
9E,Endeavor Air Inc.,Endeavor Air
AA,American Airlines Inc.,American
AS,Alaska Airlines Inc.,Alaska
B6,JetBlue Airways,JetBlue Airways
DL,Delta Air Lines Inc.,Delta
EV,ExpressJet Airlines Inc.,ExpressJet
F9,Frontier Airlines Inc.,Frontier
FL,AirTran Airways Corporation,AirTran Airways
HA,Hawaiian Airlines Inc.,Hawaiian
MQ,Envoy Air,Envoy Air


### Python: String Functions

In [0]:
from pyspark.sql.functions import trim, upper, lower, initcap, regexp_replace

messy = spark.sql("SELECT * FROM messy_names")

# Clean names
clean = messy.select(
    col("name").alias("original"),
    trim(col("name")).alias("trimmed"),
    initcap(trim(regexp_replace(col("name"), "\\s+", " "))).alias("clean_name"),
    lower(trim(col("email"))).alias("clean_email"),
    initcap(trim(col("city"))).alias("clean_city")
)
display(clean)

original,trimmed,clean_name,clean_email,clean_city
John Smith,John Smith,John Smith,john.smith@email.com,New York
JANE DOE,JANE DOE,Jane Doe,jane.doe@email.com,New York
bob johnson,bob johnson,Bob Johnson,bob.johnson@email.com,New York
Alice Williams,Alice Williams,Alice Williams,alice.w@email.com,Los Angeles
CHARLIE brown,CHARLIE brown,Charlie Brown,charlie@email.com,Los Angeles


### String Functions Reference

| Operation | SQL | Python |
|-----------|-----|--------|
| Remove leading/trailing spaces | `TRIM(col)` | `trim(col)` |
| Remove leading spaces | `LTRIM(col)` | `ltrim(col)` |
| Remove trailing spaces | `RTRIM(col)` | `rtrim(col)` |
| Uppercase | `UPPER(col)` | `upper(col)` |
| Lowercase | `LOWER(col)` | `lower(col)` |
| Title case | `INITCAP(col)` | `initcap(col)` |
| Replace pattern | `REGEXP_REPLACE(col, pattern, replacement)` | `regexp_replace(col, pattern, replacement)` |
| Substring | `SUBSTRING(col, start, length)` | `substring(col, start, length)` |
| String length | `LENGTH(col)` | `length(col)` |
| Concatenate | `CONCAT(a, b)` | `concat(a, b)` |

## Part 5: Outlier Detection

Outliers are extreme values that may be errors or genuinely unusual observations. Two common
methods for detecting them:

### Method 1: IQR (Interquartile Range)

Values below Q1 - 1.5×IQR or above Q3 + 1.5×IQR are flagged as outliers. This method works
well for skewed data.

**SQL — IQR on diamond prices:**

In [0]:
%sql
WITH bounds AS (
    SELECT
        PERCENTILE(price, 0.25) AS q1,
        PERCENTILE(price, 0.75) AS q3,
        PERCENTILE(price, 0.75) - PERCENTILE(price, 0.25) AS iqr
    FROM diamonds
)
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN price < q1 - 1.5 * iqr THEN 1 ELSE 0 END) AS low_outliers,
    SUM(CASE WHEN price > q3 + 1.5 * iqr THEN 1 ELSE 0 END) AS high_outliers,
    ROUND(q1 - 1.5 * iqr, 2) AS lower_bound,
    ROUND(q3 + 1.5 * iqr, 2) AS upper_bound
FROM diamonds, bounds
GROUP BY q1, q3, iqr

total,low_outliers,high_outliers,lower_bound,upper_bound
53940,0,3540,-5611.38,11885.63


**Python — IQR on diamond prices:**

In [0]:
diamonds = spark.table("diamonds")

# Calculate IQR
q1, q3 = diamonds.approxQuantile("price", [0.25, 0.75], 0.01)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Q1: {q1}, Q3: {q3}, IQR: {iqr}")
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")

# Flag outliers
from pyspark.sql.functions import when

diamonds_flagged = diamonds.withColumn(
    "is_outlier",
    when((col("price") < lower_bound) | (col("price") > upper_bound), True).otherwise(False)
)

print(f"Outliers: {diamonds_flagged.filter(col('is_outlier')).count()} out of {diamonds.count()}")

Q1: 932.0, Q3: 5280.0, IQR: 4348.0
Lower bound: -5590.0, Upper bound: 11802.0
Outliers: 3605 out of 53940


### Method 2: Z-Score

Values with |z-score| > 3 are flagged as outliers. This method assumes roughly normal
distribution — it works best on symmetric data. We used z-scores in Week 3.

**SQL — Z-score on flight arrival delays:**

In [0]:
%sql
WITH stats AS (
    SELECT
        AVG(arr_delay) AS mean_delay,
        STDDEV(arr_delay) AS std_delay
    FROM flights
    WHERE arr_delay IS NOT NULL
)
SELECT
    carrier, flight, arr_delay,
    ROUND((arr_delay - mean_delay) / std_delay, 2) AS z_score
FROM flights, stats
WHERE arr_delay IS NOT NULL
  AND ABS((arr_delay - mean_delay) / std_delay) > 3
ORDER BY z_score DESC
LIMIT 15

carrier,flight,arr_delay,z_score
HA,51,1272,28.34
MQ,3535,1127,25.1
MQ,3695,1109,24.69
AA,177,1007,22.41
MQ,3075,989,22.0
DL,2391,931,20.7
DL,2119,915,20.35
DL,2047,895,19.9
AA,172,878,19.52
MQ,3744,875,19.45


**Python — Z-score on flight arrival delays:**

In [0]:
from pyspark.sql.functions import mean as spark_mean, stddev as spark_stddev, abs as spark_abs, round

flights = spark.table("flights").filter(col("arr_delay").isNotNull())

# Calculate mean and std
stats = flights.select(
    spark_mean("arr_delay").alias("mean_delay"),
    spark_stddev("arr_delay").alias("std_delay")
).collect()[0]

mean_delay = stats["mean_delay"]
std_delay = stats["std_delay"]

# Add z-score and flag outliers
flights_z = flights.withColumn(
    "z_score", round((col("arr_delay") - mean_delay) / std_delay, 2)
).withColumn(
    "is_outlier", spark_abs(col("z_score")) > 3
)

print(f"Outliers (|z| > 3): {flights_z.filter(col('is_outlier')).count()}")
display(flights_z.filter(col("is_outlier")).select("carrier", "flight", "arr_delay", "z_score").orderBy(col("z_score").desc()).limit(15))

Outliers (|z| > 3): 7167


carrier,flight,arr_delay,z_score
HA,51,1272,28.34
MQ,3535,1127,25.1
MQ,3695,1109,24.69
AA,177,1007,22.41
MQ,3075,989,22.0
DL,2391,931,20.7
DL,2119,915,20.35
DL,2047,895,19.9
AA,172,878,19.52
MQ,3744,875,19.45


### IQR vs Z-Score Comparison

| | IQR Method | Z-Score Method |
|---|-----------|----------------|
| Best for | Skewed data (prices, incomes) | Roughly symmetric data |
| Threshold | Q1 - 1.5×IQR / Q3 + 1.5×IQR | \|z\| > 2 or 3 |
| Assumes normality? | No | Yes |
| Sensitive to extreme values? | Less sensitive | More sensitive |
| Use when | Distribution is unknown or skewed | You know data is roughly normal |

## Part 6: Data Validation Checks

After cleaning, run validation checks to confirm your data meets expectations. Think of these
as automated sanity checks.

### SQL: Validation Report

In [0]:
%sql
-- Validation report with PASS/FAIL for multiple rules
SELECT 'Row count > 300000' AS check_name,
       CASE WHEN COUNT(*) > 300000 THEN 'PASS' ELSE 'FAIL' END AS result,
       CAST(COUNT(*) AS STRING) AS detail
FROM flights

UNION ALL

SELECT 'No NULL carrier' AS check_name,
       CASE WHEN SUM(CASE WHEN carrier IS NULL THEN 1 ELSE 0 END) = 0 THEN 'PASS' ELSE 'FAIL' END AS result,
       CAST(SUM(CASE WHEN carrier IS NULL THEN 1 ELSE 0 END) AS STRING) AS detail
FROM flights

UNION ALL

SELECT 'All diamond cuts valid' AS check_name,
       CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END AS result,
       CAST(COUNT(*) AS STRING) || ' invalid rows' AS detail
FROM diamonds
WHERE cut NOT IN ('Fair', 'Good', 'Very Good', 'Premium', 'Ideal')

UNION ALL

SELECT 'Diamond price > 0' AS check_name,
       CASE WHEN SUM(CASE WHEN price <= 0 THEN 1 ELSE 0 END) = 0 THEN 'PASS' ELSE 'FAIL' END AS result,
       CAST(SUM(CASE WHEN price <= 0 THEN 1 ELSE 0 END) AS STRING) || ' invalid rows' AS detail
FROM diamonds

UNION ALL

SELECT 'arr_delay missing < 5%' AS check_name,
       CASE WHEN 100.0 * SUM(CASE WHEN arr_delay IS NULL THEN 1 ELSE 0 END) / COUNT(*) < 5 THEN 'PASS' ELSE 'FAIL' END AS result,
       ROUND(100.0 * SUM(CASE WHEN arr_delay IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) || '%' AS detail
FROM flights

check_name,result,detail
No NULL carrier,PASS,0
All diamond cuts valid,PASS,0 invalid rows
Diamond price > 0,PASS,0 invalid rows
arr_delay missing < 5%,PASS,2.80%
Row count > 300000,PASS,336776


### Python: Assertion-Style Checks

In [0]:
flights = spark.table("flights")
diamonds = spark.table("diamonds")

# Check 1: Row count
flight_count = flights.count()
status = "PASS" if flight_count > 300000 else "FAIL"
print(f"[{status}] Flight row count > 300,000: {flight_count}")

# Check 2: No NULL carriers
null_carriers = flights.filter(col("carrier").isNull()).count()
status = "PASS" if null_carriers == 0 else "FAIL"
print(f"[{status}] No NULL carrier values: {null_carriers} nulls found")

# Check 3: Valid diamond cuts
valid_cuts = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
invalid_cuts = diamonds.filter(~col("cut").isin(valid_cuts)).count()
status = "PASS" if invalid_cuts == 0 else "FAIL"
print(f"[{status}] All diamond cuts valid: {invalid_cuts} invalid rows")

# Check 4: Positive diamond prices
bad_prices = diamonds.filter(col("price") <= 0).count()
status = "PASS" if bad_prices == 0 else "FAIL"
print(f"[{status}] All diamond prices > 0: {bad_prices} invalid rows")

# Check 5: Missing arr_delay under threshold
total = flights.count()
missing = flights.filter(col("arr_delay").isNull()).count()
pct = 100.0 * missing / total
status = "PASS" if pct < 5 else "FAIL"
print(f"[{status}] arr_delay missing < 5%: {pct:.2f}%")

[PASS] Flight row count > 300,000: 336776
[PASS] No NULL carrier values: 0 nulls found
[PASS] All diamond cuts valid: 0 invalid rows
[PASS] All diamond prices > 0: 0 invalid rows
[PASS] arr_delay missing < 5%: 2.80%


---

*We'll now shift from data cleaning to version control. Take a stretch break if you need one!*

---

## Part 7: Git & GitHub Fundamentals

Git tracks changes to your files over time. GitHub stores those tracked files online so you can
share them and never lose work.

### Key Concepts

| Git Concept | Real-World Analogy | What It Does |
|-------------|-------------------|--------------|
| Repository (repo) | A project folder with memory | Stores all files and their full change history |
| Commit | Saving a snapshot with a sticky note | Records what changed and why |
| Branch | A parallel draft of your document | Lets you experiment without affecting the main version |
| Push | Uploading your saved work to the cloud | Sends your local commits to GitHub |
| Pull | Downloading the latest version | Gets changes others have pushed |
| Clone | Photocopying the entire folder | Creates a local copy of a remote repo |

You don't need to memorize Git command-line syntax for this course — Databricks provides a visual
interface for all Git operations.

## Part 8: Creating a GitHub Repository

If you created a GitHub account in Week 1, you're ready. If not, go back to the Week 1 guide
and sign up first.

### Steps:

1. Go to [https://github.com](https://github.com) and log in
2. Click the **+** icon in the top right corner and select **"New repository"**
3. Fill in the repository details:
   - **Repository name**: `empirical-research-notebooks`
   - **Description**: "Databricks notebooks for Foundations of Empirical Research"
   - **Visibility**: Public (so the instructor can see your work)
   - Check **"Add a README file"**
4. Click **"Create repository"**
5. You should now see your new repository page with a README.md file

Keep this browser tab open — you'll need the repo URL in the next step.

## Part 9: Connecting Databricks to GitHub (Repos)

Databricks Repos lets you clone a GitHub repository directly into your workspace, so notebooks
are version-controlled automatically.

### Step 1: Generate a GitHub Personal Access Token (PAT)

A PAT is a password specifically for apps to access your GitHub account. Databricks Free Edition
requires a PAT (not OAuth) for authentication.

1. On GitHub, click your **profile picture** (top right) → **Settings**
2. Scroll down in the left sidebar and click **"Developer settings"**
3. Click **"Personal access tokens"** → **"Tokens (classic)"**
4. Click **"Generate new token"** → **"Generate new token (classic)"**
5. Fill in the token settings:
   - **Note**: "Databricks access"
   - **Expiration**: 90 days (you can regenerate later)
   - **Scopes**: Check **`repo`** (this grants full access to your repositories)
6. Click **"Generate token"**
7. **Copy the token immediately** — you won't be able to see it again. Save it somewhere safe.

### Step 2: Link GitHub in Databricks Settings

1. In Databricks, click your **username** in the top right → **Settings**
2. Click **"Linked accounts"** (or **"Git integration"** depending on your version)
3. Set **Git provider** to **GitHub**
4. Paste your **Personal Access Token** in the token field
5. Enter your **GitHub username**
6. Click **Save**

### Step 3: Clone Your Repository into Databricks

1. In the Databricks sidebar, click **"Repos"** (or **"Workspace"** → **"Repos"**)
2. Click **"Add Repo"**
3. Paste your repository URL: `https://github.com/YOUR-USERNAME/empirical-research-notebooks`
4. Click **"Create Repo"**
5. You should now see your repository folder in the Repos section with the README.md file

### Step 4: Create a Notebook Inside the Repo

1. Right-click on your repo folder in the Repos section
2. Select **"Create"** → **"Notebook"**
3. Name it `week-6-practice` and select **Python** as the default language
4. This notebook is now tracked by Git — any changes you save can be committed and pushed.

## Part 10: Committing and Pushing Changes

After making changes to a notebook in your repo, you need to commit (save a snapshot) and push
(upload to GitHub).

### Steps:

1. Click the **Git status icon** next to your repo name in the sidebar
2. You'll see a list of **modified files** — check the boxes next to the files you want to commit
3. Write a **commit message** describing what changed
4. Click **"Commit & Push"**

### Commit Message Best Practices

| Bad Message | Good Message | Why It's Better |
|-------------|-------------|-----------------|
| "update" | "Add filtering examples to week-3 notebook" | Says what was added and where |
| "fixed stuff" | "Fix NULL handling in delay calculation" | Names the specific fix |
| "changes" | "Add diamond price histogram using matplotlib" | Describes the actual work |
| "done" | "Complete homework 2: aggregation exercises" | Identifies which assignment |

A good commit message answers: **"If I read this in 3 months, will I know what this change was about?"**

### Pulling Changes

If you edit files directly on GitHub (like the README), pull those changes into Databricks:

1. Click the **Git status icon** next to your repo
2. Click **"Pull"** to download the latest version from GitHub

## Part 11: Branching Basics

Branches let you experiment without affecting your main work. Think of it as making a copy of your
notebook to try something risky.

### Create a Branch from Databricks UI

1. Click the **Git status icon** next to your repo
2. Click the **branch dropdown** (it will say "main")
3. Click **"Create Branch"**
4. Name it something descriptive like `homework-6-draft`
5. Make your changes and commit to this branch
6. When you're happy with the changes, go to GitHub.com and create a **Pull Request** to merge
   the branch back into main

### Pull Requests (Brief Intro)

A Pull Request (PR) on GitHub is a way to propose merging your branch into main. It shows all
your changes in one view, and in team settings, others can review before merging. For this course,
you'll create PRs mostly to practice the workflow.

To create a PR: go to your repo on GitHub → click **"Pull requests"** → **"New pull request"** →
select your branch → **"Create pull request"**.

## Quick Reference Tables

### Data Cleaning Operations

| Operation | SQL | Python |
|-----------|-----|--------|
| Find duplicates | `GROUP BY ... HAVING COUNT(*) > 1` | `groupBy(...).count().filter(col("count") > 1)` |
| Remove duplicates | `ROW_NUMBER() OVER (...) + WHERE rn = 1` | `.dropDuplicates([cols])` |
| Count NULLs | `SUM(CASE WHEN col IS NULL THEN 1 ELSE 0 END)` | `count(when(col.isNull(), 1))` |
| Drop NULL rows | `WHERE col IS NOT NULL` | `.dropna(subset=["col"])` |
| Fill NULLs | `COALESCE(col, value)` | `.fillna({"col": value})` |
| Check schema | `DESCRIBE table` | `.printSchema()` |
| Cast type | `CAST(col AS TYPE)` | `.cast(Type())` |
| Trim spaces | `TRIM(col)` | `trim(col)` |
| Change case | `UPPER()` / `LOWER()` / `INITCAP()` | `upper()` / `lower()` / `initcap()` |
| Regex replace | `REGEXP_REPLACE(col, pattern, repl)` | `regexp_replace(col, pattern, repl)` |
| IQR outliers | `PERCENTILE(col, 0.25/0.75)` | `.approxQuantile("col", [0.25, 0.75], 0.01)` |
| Z-score outliers | `(val - AVG) / STDDEV` | `(col - mean) / std` |

### Git & Databricks Repos

| Action | How To |
|--------|--------|
| Create GitHub repo | GitHub.com → **+** → New repository |
| Generate PAT | GitHub → Settings → Developer settings → Personal access tokens |
| Link GitHub to Databricks | Databricks → Settings → Linked accounts → paste PAT |
| Clone repo | Databricks → Repos → Add Repo → paste URL |
| Create notebook in repo | Right-click repo folder → Create → Notebook |
| Commit & push | Click Git status icon → check files → write message → Commit & Push |
| Pull changes | Click Git status icon → Pull |
| Create branch | Git status icon → branch dropdown → Create Branch |
| Create Pull Request | GitHub.com → Pull requests → New pull request |

## Week 6 Checklist

- [ ] Found duplicate rows using GROUP BY ... HAVING
- [ ] Audited NULL values across columns with percentages
- [ ] Cleaned messy strings with TRIM, INITCAP, REGEXP_REPLACE
- [ ] Detected outliers using IQR or z-score method
- [ ] Ran a validation check with PASS/FAIL output
- [ ] Created a GitHub repository (`empirical-research-notebooks`)
- [ ] Generated a Personal Access Token (PAT)
- [ ] Linked GitHub to Databricks via Settings
- [ ] Cloned the repo into Databricks Repos
- [ ] Made a commit with a descriptive message

## Homework

1. **Push Your Notebooks to GitHub**
   - Push at least 2 previous notebooks (from any week) to your `empirical-research-notebooks` repo
   - Use descriptive commit messages for each (not "upload" or "added files")
   - Share your repo URL on Slack in the course channel

2. **Missing Value Investigation**
   - Calculate the percentage of missing values for every delay-related column in `flights`
     (dep_delay, arr_delay, air_time)
   - For `arr_delay`, compare: what is the mean vs the median of the non-null values? Which would
     be a better fill value and why? Write your answer as a comment.

3. **Full Cleaning Pipeline**
   - Starting with `flights`, build a cleaning pipeline that:
     - (a) Removes duplicate flight records (by year, month, day, carrier, flight)
     - (b) Fills missing `dep_delay` with the median value
     - (c) Flags outliers in `arr_delay` using the IQR method
     - (d) Runs at least 3 validation checks on the cleaned result
   - Complete in SQL or Python (bonus: do both)

4. **Share Your Validation Output**
   - Take a screenshot of your PASS/FAIL validation report output
   - Post it on Slack with one sentence about something that surprised you about the data quality

# Completed Homework: Week 6

**Student:** Kerra Sinanan  
**Topic:** Data Cleaning & Validation + GitHub & Version Control  

This section completes the Week 6 homework requirements. It keeps the original lesson above intact and adds the required missing-value investigation, cleaning pipeline, validation report, and Slack/GitHub submission text.

## Homework 1: Push Notebooks to GitHub

**Repository name:** `empirical-research-notebooks`

**Recommended files to push:**
1. This completed Week 6 notebook
2. One previous Databricks notebook from an earlier week

**Descriptive commit messages to use:**
- `Complete Week 6 data cleaning and validation notebook`
- `Add previous empirical research Databricks notebook`


## Homework 2: Missing Value Investigation

Requirement:
- Calculate the percentage of missing values for every delay-related column in `flights`.
- Compare the mean vs median for non-null `arr_delay`.
- Explain which fill value is better.

In [0]:
from pyspark.sql.functions import col, count, when, round as spark_round, mean as spark_mean, expr

flights = spark.table("flights")

delay_columns = ["dep_delay", "arr_delay", "air_time"]

missing_value_report = flights.select(
    count("*").alias("total_rows"),
    *[
        count(when(col(c).isNull(), c)).alias(f"missing_{c}")
        for c in delay_columns
    ],
    *[
        spark_round(
            100.0 * count(when(col(c).isNull(), c)) / count("*"),
            2
        ).alias(f"pct_missing_{c}")
        for c in delay_columns
    ]
)

display(missing_value_report)

total_rows,missing_dep_delay,missing_arr_delay,missing_air_time,pct_missing_dep_delay,pct_missing_arr_delay,pct_missing_air_time
336776,8255,9430,9430,2.45,2.8,2.8


In [0]:
# Mean vs median for arr_delay using only non-null values
arr_delay_stats = flights.filter(col("arr_delay").isNotNull()).select(
    spark_round(spark_mean("arr_delay"), 2).alias("mean_arr_delay"),
    expr("percentile_approx(arr_delay, 0.5)").alias("median_arr_delay")
)

display(arr_delay_stats)

mean_arr_delay,median_arr_delay
6.9,-5


### Homework 2 Written Answer

For `arr_delay`, the **median** is the better fill value because flight delays are usually skewed by a smaller number of extreme late flights. The mean gets pulled upward by those unusual delays, while the median gives a more typical value for a normal flight. I would use the median to fill missing `arr_delay` if the goal is to avoid exaggerating average delay behavior.

## Homework 3: Full Cleaning Pipeline in Python

This pipeline:
1. Removes duplicate flight records by `year`, `month`, `day`, `carrier`, and `flight`
2. Fills missing `dep_delay` with the median value
3. Flags outliers in `arr_delay` using the IQR method
4. Runs validation checks on the cleaned result

In [0]:
from pyspark.sql.functions import lit

# Start with the original flights table
flights_raw = spark.table("flights")

# Step A: Remove duplicate flight records by the assignment key
flight_key = ["year", "month", "day", "carrier", "flight"]

flights_deduped = flights_raw.dropDuplicates(flight_key)

print(f"Original row count: {flights_raw.count()}")
print(f"After duplicate removal: {flights_deduped.count()}")

Original row count: 336776
After duplicate removal: 336752


In [0]:
# Step B: Fill missing dep_delay with the median value
dep_delay_median = flights_deduped.filter(col("dep_delay").isNotNull()).approxQuantile(
    "dep_delay",
    [0.5],
    0.01
)[0]

flights_dep_filled = flights_deduped.fillna({"dep_delay": dep_delay_median})

print(f"Median dep_delay used for fill: {dep_delay_median}")

Median dep_delay used for fill: -1.0


In [0]:
# Step C: Flag arr_delay outliers using the IQR method
arr_delay_non_null = flights_dep_filled.filter(col("arr_delay").isNotNull())

q1, q3 = arr_delay_non_null.approxQuantile("arr_delay", [0.25, 0.75], 0.01)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

flights_cleaned_week6 = flights_dep_filled.withColumn(
    "arr_delay_outlier",
    when(col("arr_delay").isNull(), False)
    .when((col("arr_delay") < lower_bound) | (col("arr_delay") > upper_bound), True)
    .otherwise(False)
).withColumn(
    "arr_delay_iqr_lower_bound",
    lit(float(lower_bound))
).withColumn(
    "arr_delay_iqr_upper_bound",
    lit(float(upper_bound))
)

print(f"arr_delay Q1: {q1}")
print(f"arr_delay Q3: {q3}")
print(f"arr_delay IQR: {iqr}")
print(f"Lower outlier bound: {lower_bound}")
print(f"Upper outlier bound: {upper_bound}")

display(
    flights_cleaned_week6
    .select(
        "year", "month", "day", "carrier", "flight",
        "dep_delay", "arr_delay", "arr_delay_outlier",
        "arr_delay_iqr_lower_bound", "arr_delay_iqr_upper_bound"
    )
    .orderBy(col("arr_delay").desc())
    .limit(20)
)

arr_delay Q1: -16.0
arr_delay Q3: 14.0
arr_delay IQR: 30.0
Lower outlier bound: -61.0
Upper outlier bound: 59.0


year,month,day,carrier,flight,dep_delay,arr_delay,arr_delay_outlier,arr_delay_iqr_lower_bound,arr_delay_iqr_upper_bound
2013,1,9,HA,51,1301,1272,true,-61.0,59.0
2013,6,15,MQ,3535,1137,1127,true,-61.0,59.0
2013,1,10,MQ,3695,1126,1109,true,-61.0,59.0
2013,9,20,AA,177,1014,1007,true,-61.0,59.0
2013,7,22,MQ,3075,1005,989,true,-61.0,59.0
2013,4,10,DL,2391,960,931,true,-61.0,59.0
2013,3,17,DL,2119,911,915,true,-61.0,59.0
2013,7,22,DL,2047,898,895,true,-61.0,59.0
2013,12,5,AA,172,896,878,true,-61.0,59.0
2013,5,3,MQ,3744,878,875,true,-61.0,59.0


In [0]:
# Create a temporary view so the cleaned result can also be queried in SQL
flights_cleaned_week6.createOrReplaceTempView("flights_cleaned_week6")

# flights_cleaned_week6.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("flights_cleaned_week6")

## Homework 3: Validation Report Output



In [0]:
# Validation calculations
cleaned_count = flights_cleaned_week6.count()

duplicate_keys_after = (
    flights_cleaned_week6
    .groupBy(flight_key)
    .count()
    .filter(col("count") > 1)
    .count()
)

missing_dep_delay_after = flights_cleaned_week6.filter(col("dep_delay").isNull()).count()
missing_carrier_after = flights_cleaned_week6.filter(col("carrier").isNull()).count()
outlier_count = flights_cleaned_week6.filter(col("arr_delay_outlier") == True).count()

validation_rows = [
    (
        "Cleaned row count is positive",
        "PASS" if cleaned_count > 0 else "FAIL",
        f"{cleaned_count} cleaned rows"
    ),
    (
        "No duplicate records by year/month/day/carrier/flight",
        "PASS" if duplicate_keys_after == 0 else "FAIL",
        f"{duplicate_keys_after} duplicate key groups"
    ),
    (
        "Missing dep_delay filled with median",
        "PASS" if missing_dep_delay_after == 0 else "FAIL",
        f"{missing_dep_delay_after} missing dep_delay values"
    ),
    (
        "No NULL carrier values",
        "PASS" if missing_carrier_after == 0 else "FAIL",
        f"{missing_carrier_after} missing carrier values"
    ),
    (
        "arr_delay outlier flag created",
        "PASS" if "arr_delay_outlier" in flights_cleaned_week6.columns else "FAIL",
        f"{outlier_count} rows flagged as outliers"
    ),
    (
        "IQR bounds are valid",
        "PASS" if lower_bound < upper_bound else "FAIL",
        f"lower={lower_bound:.2f}, upper={upper_bound:.2f}"
    )
]

validation_report = spark.createDataFrame(
    validation_rows,
    ["check_name", "result", "detail"]
)

display(validation_report)

check_name,result,detail
Cleaned row count is positive,PASS,336752 cleaned rows
No duplicate records by year/month/day/carrier/flight,PASS,0 duplicate key groups
Missing dep_delay filled with median,PASS,0 missing dep_delay values
No NULL carrier values,PASS,0 missing carrier values
arr_delay outlier flag created,PASS,28478 rows flagged as outliers
IQR bounds are valid,PASS,"lower=-61.00, upper=59.00"


## Homework 3 Bonus: Full Cleaning Pipeline in SQL

This is the same cleaning logic written in SQL.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW flights_cleaned_week6_sql AS
WITH numbered AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY year, month, day, carrier, flight
            ORDER BY sched_dep_time
        ) AS rn
    FROM flights
),
deduped AS (
    SELECT *
    FROM numbered
    WHERE rn = 1
),
dep_median AS (
    SELECT percentile_approx(dep_delay, 0.5) AS median_dep_delay
    FROM deduped
    WHERE dep_delay IS NOT NULL
),
dep_filled AS (
    SELECT
        d.*,
        COALESCE(d.dep_delay, m.median_dep_delay) AS dep_delay_filled
    FROM deduped d
    CROSS JOIN dep_median m
),
bounds AS (
    SELECT
        percentile_approx(arr_delay, 0.25) AS q1,
        percentile_approx(arr_delay, 0.75) AS q3,
        percentile_approx(arr_delay, 0.75) - percentile_approx(arr_delay, 0.25) AS iqr
    FROM dep_filled
    WHERE arr_delay IS NOT NULL
)
SELECT
    f.*,
    CASE
        WHEN f.arr_delay IS NULL THEN false
        WHEN f.arr_delay < b.q1 - 1.5 * b.iqr THEN true
        WHEN f.arr_delay > b.q3 + 1.5 * b.iqr THEN true
        ELSE false
    END AS arr_delay_outlier_sql,
    b.q1 - 1.5 * b.iqr AS arr_delay_iqr_lower_bound_sql,
    b.q3 + 1.5 * b.iqr AS arr_delay_iqr_upper_bound_sql
FROM dep_filled f
CROSS JOIN bounds b

In [0]:
%sql
SELECT
    carrier,
    flight,
    dep_delay,
    dep_delay_filled,
    arr_delay,
    arr_delay_outlier_sql
FROM flights_cleaned_week6_sql
ORDER BY arr_delay DESC
LIMIT 20

carrier,flight,dep_delay,dep_delay_filled,arr_delay,arr_delay_outlier_sql
HA,51,1301,1301,1272,true
MQ,3535,1137,1137,1127,true
MQ,3695,1126,1126,1109,true
AA,177,1014,1014,1007,true
MQ,3075,1005,1005,989,true
DL,2391,960,960,931,true
DL,2119,911,911,915,true
DL,2047,898,898,895,true
AA,172,896,896,878,true
MQ,3744,878,878,875,true


## Homework 4: Slack Validation Post

`Something that surprised me is that missing delay data is not automatically just bad data. In flight records, missing delay fields can signal flights with incomplete movement data, cancellations, or other operational issues, so validation matters before deciding whether to drop or fill values.`

## Completed Week 6 Checklist

- [x] Found duplicate rows using `GROUP BY ... HAVING`
- [x] Audited NULL values across delay-related columns with percentages
- [x] Compared mean vs median for `arr_delay`
- [x] Explained why median is the better fill value for skewed delay data
- [x] Built a full Python cleaning pipeline
- [x] Removed duplicate flight records by key
- [x] Filled missing `dep_delay` with the median
- [x] Flagged `arr_delay` outliers using IQR
- [x] Ran a PASS/FAIL validation report
- [x] Added bonus SQL cleaning pipeline
- [x] Included GitHub commit-message text
- [x] Included Slack post sentence for the validation screenshot
